In [1]:
import pandas as pd

In [9]:
df = pd.read_excel(
    "/Users/sreejanallamala/Downloads/offenses-known-to-le-2024/CIUS_Table_8_Offenses_Known_to_Law_Enforcement_by_State_by_City_2024.xlsx",
    skiprows=4
)

In [10]:
df.columns = [
    "state", "city", "population", "violent_crime",
    "murder", "rape", "robbery", "aggravated_assault",
    "property_crime", "burglary", "larceny_theft",
    "motor_vehicle_theft", "arson"
]

In [11]:
# Drop rows where both state and city are empty (spacer rows in the Excel)
df = df.dropna(subset=["city"])

# Fill state column downward (state name only appears on first row for each state)
df["state"] = df["state"].fillna(method="ffill")

# Convert numeric columns (they may have loaded as strings)
numeric_cols = ["population", "violent_crime", "murder", "rape", "robbery",
                "aggravated_assault", "property_crime", "burglary",
                "larceny_theft", "motor_vehicle_theft", "arson"]

df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

# Drop rows that are still mostly empty after conversion
df = df.dropna(subset=["population", "violent_crime"])

# Reset index
df = df.reset_index(drop=True)

/var/folders/py/yhh4x__x569fvjmc099x_25m0000gn/T/ipykernel_23708/570793443.py:5: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["state"] = df["state"].fillna(method="ffill")


In [12]:
print(df.shape)   # how many rows and columns
print(df.head())  # first few rows
print(df.dtypes)  # confirm columns are the right type

(8986, 13)
     state            city  population  violent_crime  murder  rape  robbery  \
0  ALABAMA      Adamsville      4111.0           23.0     1.0   1.0      0.0   
1  ALABAMA         Addison       669.0            3.0     0.0   0.0      0.0   
2  ALABAMA       Alabaster     34349.0           34.0     0.0   1.0      4.0   
3  ALABAMA     Albertville     23242.0           55.0     1.0  14.0      3.0   
4  ALABAMA  Alexander City     14353.0          147.0     4.0   7.0     11.0   

   aggravated_assault  property_crime  burglary  larceny_theft  \
0                21.0           130.0      12.0           97.0   
1                 3.0            13.0       4.0            7.0   
2                29.0           519.0      17.0          480.0   
3                37.0           409.0      65.0          299.0   
4               125.0           515.0      91.0          398.0   

   motor_vehicle_theft  arson  
0                 21.0    2.0  
1                  2.0    0.0  
2              

In [13]:
# General stats for all numeric columns
print(df.describe())

# Average violent and property crime across all cities
print(df["violent_crime"].mean())
print(df["property_crime"].mean())

# Which states have the most total violent crime?
state_summary = df.groupby("state")[["violent_crime", "property_crime"]].sum()
state_summary = state_summary.sort_values("violent_crime", ascending=False)
print(state_summary.head(10))

# How many cities per state are in the dataset?
print(df.groupby("state")["city"].count().sort_values(ascending=False))

         population  violent_crime       murder         rape       robbery  \
count  8.986000e+03    8986.000000  8986.000000  8986.000000   8986.000000   
mean   2.337559e+04     100.638326     1.328400     9.508569     19.218673   
std    1.223671e+05     885.069938    11.148893    52.960867    245.192532   
min    0.000000e+00       0.000000     0.000000     0.000000      0.000000   
25%    2.494250e+03       2.000000     0.000000     0.000000      0.000000   
50%    6.711000e+03      10.000000     0.000000     1.000000      0.000000   
75%    1.843600e+04      37.000000     0.000000     5.000000      3.000000   
max    8.299271e+06   55690.000000   461.000000  1943.000000  15559.000000   

       aggravated_assault  property_crime      burglary  larceny_theft  \
count         8986.000000     8986.000000   8986.000000    8986.000000   
mean            70.582684      497.402404     62.891164     359.788226   
std            601.560233     3388.235713    399.791606    2505.077046   
m

In [14]:
df.to_csv("crime_cleaned.csv", index=False)

In [15]:
print(df[["state", "city"]].head(10))

     state            city
0  ALABAMA      Adamsville
1  ALABAMA         Addison
2  ALABAMA       Alabaster
3  ALABAMA     Albertville
4  ALABAMA  Alexander City
5  ALABAMA      Aliceville
6  ALABAMA       Andalusia
7  ALABAMA        Anniston
8  ALABAMA            Arab
9  ALABAMA         Ardmore


In [16]:
df["violent_crime_rate"] = (df["violent_crime"] / df["population"]) * 100000
df["property_crime_rate"] = (df["property_crime"] / df["population"]) * 100000

In [17]:
# Most common type of crime overall (sum of each crime type)
crime_totals = df[["murder", "rape", "robbery", "aggravated_assault", 
                    "burglary", "larceny_theft", "motor_vehicle_theft", "arson"]].sum()
print("Most common crime type:")
print(crime_totals.sort_values(ascending=False))

# City with highest violent crime
print("\nCity with most violent crime:")
print(df.loc[df["violent_crime"].idxmax(), ["state", "city", "violent_crime"]])

# City with highest property crime
print("\nCity with most property crime:")
print(df.loc[df["property_crime"].idxmax(), ["state", "city", "property_crime"]])

# Top 10 cities for violent crime
print("\nTop 10 cities by violent crime:")
print(df[["state", "city", "violent_crime"]].sort_values("violent_crime", ascending=False).head(10))

Most common crime type:
larceny_theft          3233057.0
motor_vehicle_theft     671461.0
aggravated_assault      634256.0
burglary                565140.0
robbery                 172699.0
rape                     85444.0
arson                    25458.0
murder                   11937.0
dtype: float64

City with most violent crime:
state            NEW YORK
city             New York
violent_crime     55690.0
Name: 5261, dtype: object

City with most property crime:
state             NEW YORK
city              New York
property_crime    196549.0
Name: 5261, dtype: object

Top 10 cities by violent crime:
             state          city  violent_crime
5261      NEW YORK      New York        55690.0
786     CALIFORNIA  Los Angeles2        27656.0
7864         TEXAS       Houston        26628.0
7510     TENNESSEE       Memphis        15338.0
1727      ILLINOIS       Chicago        14245.0
6897  PENNSYLVANIA  Philadelphia        14078.0
324        ARIZONA       Phoenix        13296.0
3419  